In [1]:
from sklearn.base import BaseEstimator, TransformerMixin

class CreditFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, bureau_path="../data/bureau.csv"):
        self.bureau_path = bureau_path
        self.bureau_agg_ = None

    def fit(self, X, y=None):
        bureau = pd.read_csv(self.bureau_path)
        bureau.columns = bureau.columns.str.strip()

        bureau_agg = bureau.groupby("SK_ID_CURR").agg(
            num_bureau_records=("SK_ID_BUREAU", "count"),
            num_active_credits=("CREDIT_ACTIVE", lambda s: (s == "Active").sum()),
            mean_days_credit=("DAYS_CREDIT", "mean"),
            max_overdue=("AMT_CREDIT_SUM_OVERDUE", "max")
        ).reset_index()

        self.bureau_agg_ = bureau_agg
        return self

    def transform(self, X):
        df = X.copy()
        df.columns = df.columns.str.strip()

        if "DAYS_EMPLOYED" in df.columns:
            df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

        if "DAYS_BIRTH" in df.columns:
            df["age"] = (-df["DAYS_BIRTH"]) / 365

        if "AMT_INCOME_TOTAL" in df.columns:
            df["AMT_INCOME_TOTAL"] = df["AMT_INCOME_TOTAL"].replace(0, np.nan)

        if {"AMT_CREDIT", "AMT_INCOME_TOTAL"}.issubset(df.columns):
            df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]

        if {"AMT_ANNUITY", "AMT_INCOME_TOTAL"}.issubset(df.columns):
            df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]

        if {"AMT_ANNUITY", "AMT_CREDIT"}.issubset(df.columns):
            df["CREDIT_TERM"] = df["AMT_ANNUITY"] / df["AMT_CREDIT"]

        if {"DAYS_EMPLOYED", "DAYS_BIRTH"}.issubset(df.columns):
            df["DAYS_EMPLOYED_RATIO"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]

        if {"AMT_INCOME_TOTAL", "CNT_FAM_MEMBERS"}.issubset(df.columns):
            df["INCOME_PER_PERSON"] = df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]

        ext_cols = [c for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if c in df.columns]
        if len(ext_cols) > 0:
            df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
            df["EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)

        if self.bureau_agg_ is not None and "SK_ID_CURR" in df.columns:
            df = df.merge(self.bureau_agg_, on="SK_ID_CURR", how="left")

        return df

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBClassifier

import pandas as pd
import numpy as np

raw = pd.read_csv("../data/application_train.csv")
raw.columns = raw.columns.str.strip()

y = raw["TARGET"]
X_raw = raw.drop(columns=["TARGET"])

scale_pos_weight = (y == 0).sum() / (y == 1).sum()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, make_column_selector(dtype_include=np.number)),
    ("cat", categorical_pipe, make_column_selector(dtype_include=object))
])

pipeline = Pipeline([
    ("features", CreditFeatureEngineer(bureau_path="../data/bureau.csv")),
    ("prep", preprocessor),
    ("model", XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ))
])

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

pipeline.fit(X_train_raw, y_train_raw)

proba = pipeline.predict_proba(X_test_raw)[:, 1]
auc = roc_auc_score(y_test_raw, proba)
gini = 2 * auc - 1
fpr, tpr, _ = roc_curve(y_test_raw, proba)
ks = np.max(tpr - fpr)

print("AUROC:", auc)
print("Gini:", gini)
print("KS:", ks)

AUROC: 0.7653823964326038
Gini: 0.5307647928652075
KS: 0.397673676469661


In [19]:
import joblib

joblib.dump(pipeline, "../models/credit_scoring_pipeline.pkl")
print("Saved pipeline.")

Saved pipeline.
